# Base Rates and Statistical Properties

The claims audit (notebook 04) established *which* popular numerical claims reproduce and
under which counting method. This notebook establishes whether reproducing is even
*surprising*: the base rates of count equality and "meaningful number" hits in this corpus.

Three analyses:

1. **Count-collision base rates** — how common is it for two lemmas (or roots) to occur
   exactly equally often? How many other words share each celebrated count value?
2. **Zipf fit and tail statistics** — the shape of the frequency distribution, which is what
   makes collisions abundant.
3. **Near-miss sensitivity** — for each audited claim, how many of the 13 methods land
   exactly on the claimed number vs. miss it by 1 or 2.

Honesty note, mirrored from the README: results here can conclude "not surprising under these
assumptions" or "surprising"; they cannot establish or refute intent. A planned Monte Carlo
null model (random frequency-matched pairs through the full audit machinery) is tracked as
future work, as is recounting under independent annotation schemes (QuranMorph, the corpus
website's current revision).

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from src.parser import load_morphology
from src.buckwalter import bw_to_arabic

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 250)

df = load_morphology()
assert len(df) == 77915

lemma_counts = df.LEM.value_counts()
root_counts = df.ROOT.value_counts()
print(f"distinct lemmas: {len(lemma_counts):,}   distinct roots: {len(root_counts):,}")

distinct lemmas: 4,832   distinct roots: 1,642


## 1. Count-collision base rates

A "numerical balance" claim is, mechanically, the observation that two words have equal
counts. The base-rate question: **in a vocabulary this size with this frequency distribution,
how many equal-count pairs exist in total — and how many other words sit at each celebrated
count value?**

`multiplicity(n)` = number of lemmas occurring exactly *n* times. Any multiplicity *k* yields
*k·(k−1)/2* equal-count pairs at that value.

In [2]:
mult = lemma_counts.value_counts().sort_index()      # count value -> number of lemmas with it
root_mult = root_counts.value_counts().sort_index()

hapax = int((lemma_counts == 1).sum())
tail5 = int((lemma_counts <= 5).sum())
print(f"hapax lemmas (count = 1): {hapax} ({hapax/len(lemma_counts):.0%})")
print(f"lemmas with count <= 5 : {tail5} ({tail5/len(lemma_counts):.0%})")

def equal_pairs(counts, min_count):
    m = counts[counts >= min_count].value_counts()
    return int(sum(k * (k - 1) // 2 for k in m))

rows = []
for floor in [1, 5, 10, 20, 50]:
    rows.append({"min count": floor,
                 "lemmas": int((lemma_counts >= floor).sum()),
                 "equal-count lemma pairs": equal_pairs(lemma_counts, floor),
                 "roots": int((root_counts >= floor).sum()),
                 "equal-count root pairs": equal_pairs(root_counts, floor)})
display(pd.DataFrame(rows).set_index("min count"))
print("Equal counts are abundant at every frequency floor — the raw material of the genre.")

hapax lemmas (count = 1): 1994 (41%)
lemmas with count <= 5 : 3588 (74%)


,lemmas,equal-count lemma pairs,roots,equal-count root pairs
min count,,,,
1,4832,2424642,1642,118584
5,1431,45658,833,9643
10,855,8594,594,2817
20,507,1733,411,895
50,227,181,203,106


Equal counts are abundant at every frequency floor — the raw material of the genre.


### 1.1 Who else sits at each celebrated count value?

For every count value behind a "holds" verdict in the claims audit: how many lemmas (or
roots, for root-level holds) share it, and which. A celebrated equality is one draw from
the pool listed here.

In [3]:
def lemmas_at(n, counts=lemma_counts):
    lems = counts[counts == n].index.tolist()
    return ", ".join(f"{l} [{bw_to_arabic(l)}]" for l in lems)

CELEBRATED_LEMMA = [
    (115, "dunya = akhira(F)"), (88, "malak = shaytan"), (25, "Adam = Isa; lisan claim"),
    (114, "rahma"), (12, "shahr singular; usr root"), (24, "rajul = imra'a singular"),
    (332, "qul = qalu"), (21, "shahr lemma"), (45, "iman lemma"),
]
rows = []
for n, claim in CELEBRATED_LEMMA:
    k = int(mult.get(n, 0))
    rows.append({"count": n, "claim using it": claim, "lemmas at this count": k,
                 "possible equal pairs": k * (k - 1) // 2,
                 "who": lemmas_at(n) if k <= 14 else f"({k} lemmas)"})
display(pd.DataFrame(rows).set_index("count"))

CELEBRATED_ROOT = [(50, "naf' = fasad"), (16, "jahr = 'alaniya"), (25, "lisan = maw'iza"),
                   (167, "sayyi'at side"), (234, "maghfira side")]
rows = []
for n, claim in CELEBRATED_ROOT:
    k = int(root_mult.get(n, 0))
    rows.append({"count": n, "claim using it": claim, "roots at this count": k,
                 "possible equal pairs": k * (k - 1) // 2,
                 "who": lemmas_at(n, root_counts) if k <= 14 else f"({k} roots)"})
display(pd.DataFrame(rows).set_index("count"))

,claim using it,lemmas at this count,possible equal pairs,who
count,,,,
115,dunya = akhira(F),1,0,d~unoyaA [دُّنْيَا]
88,malak = shaytan,4,6,"$ayoTa`n [شَيْطَٰن], maval [مَثَل], faEala [فَعَلَ], malak [مَلَك]"
25,Adam = Isa; lisan claim,12,66,"ya$oEuru [يَشْعُرُ], xalaA [خَلَا], m~iyva`q [مِّيثَٰق], A^dam [ا^دَم], EiysaY [عِيسَى..."
114,rahma,1,0,raHomap [رَحْمَة]
12,shahr singular; usr root,50,1225,(50 lemmas)
24,rajul = imra'a singular,13,78,"<iy~aA [إِيَّا], kaAda [كَادَ], TaEaAm [طَعَام], waraA^' [وَرَا^ء], Hay~ [حَيّ], >unva..."
332,qul = qalu,1,0,rasuwl [رَسُول]
21,shahr lemma,19,171,(19 lemmas)
45,iman lemma,7,21,"Sira`T [صِرَٰط], qadiyr [قَدِير], <iyma`n [إِيمَٰن], Hakama [حَكَمَ], >uwliY [أُولِى],..."


,claim using it,roots at this count,possible equal pairs,who
count,,,,
50,naf' = fasad,3,3,"fsd [فسد], Tyb [طيب], nfE [نفع]"
16,jahr = 'alaniya,14,91,"jhr [جهر], Eln [علن], lbb [لبب], xwn [خون], mhd [مهد], HbT [حبط], xbv [خبث], gdw [غدو]..."
25,lisan = maw'iza,13,78,"$ry [شري], Hwl [حول], sqy [سقي], wEZ [وعظ], HDr [حضر], flk [فلك], bTn [بطن], Hdd [حدد]..."
167,sayyi'at side,2,1,"kvr [كثر], swA [سوا]"
234,maghfira side,1,0,gfr [غفر]


### 1.2 The honest summary statistic

The probability-flavored way to say it: pick any two distinct lemmas with count ≥ 10 at
random; how often are they exactly equal? And the multiplicity table shows that *every*
audited equality has company at its count value or nearby. The full multiplicity table is
exported for reuse.

In [4]:
big = lemma_counts[lemma_counts >= 10]
n_pairs_total = len(big) * (len(big) - 1) // 2
n_equal = equal_pairs(lemma_counts, 10)
print(f"lemmas with count >= 10: {len(big)}")
print(f"random pair of them equal exactly: {n_equal:,} / {n_pairs_total:,} = {n_equal/n_pairs_total:.2%}")
print(f"... within +/-1: ", end="")
vals = np.sort(big.values)
within1 = sum(int(((vals >= v - 1) & (vals <= v + 1)).sum()) - 1 for v in vals) // 2
print(f"{within1:,} / {n_pairs_total:,} = {within1/n_pairs_total:.2%}")

out = pd.DataFrame({"count_value": mult.index, "n_lemmas": mult.values})
out["n_equal_pairs"] = out.n_lemmas * (out.n_lemmas - 1) // 2
out.to_csv("../output/count_multiplicity.csv", index=False)
print("\nSaved output/count_multiplicity.csv (full multiplicity table)")

lemmas with count >= 10: 855
random pair of them equal exactly: 8,594 / 365,085 = 2.35%
... within +/-1: 24,067 / 365,085 = 6.59%

Saved output/count_multiplicity.csv (full multiplicity table)


## 2. Zipf fit and tail statistics

Word frequencies in natural language follow Zipf's law: the *r*-th most frequent word has
frequency roughly proportional to 1/*r*^s with s ≈ 1, a straight line of slope −s on a
log-log rank-frequency plot. Two consequences matter here:

- a **heavy tail**: most lemmas are rare, so many lemmas get packed into each small count
  value — which is exactly where count collisions come from;
- a **steep head**: a few function words dominate the token mass.

The fit is descriptive (this corpus behaves like normal language), not evidential.

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranks = np.arange(1, len(lemma_counts) + 1)
freqs = lemma_counts.values.astype(float)

# OLS slope on log-log over the mid-range (ranks 10..1000), away from head and quantized tail
sel = (ranks >= 10) & (ranks <= 1000)
slope, intercept = np.polyfit(np.log10(ranks[sel]), np.log10(freqs[sel]), 1)

fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(ranks, freqs, ".", color="#003f5c", ms=3, label="lemmas")
rr = np.arange(1, len(root_counts) + 1)
ax.loglog(rr, root_counts.values, ".", color="#ffa600", ms=3, label="roots")
xs = np.array([10, 1000])
ax.loglog(xs, 10 ** (intercept + slope * np.log10(xs)), "-", color="#bc5090", lw=1.5,
          label=f"Zipf fit, slope = {slope:.2f} (ranks 10-1000)")
ax.set_xlabel("rank"); ax.set_ylabel("frequency")
ax.set_title("Rank-frequency distribution, Quranic Arabic Corpus v0.4 (STEM entries)")
ax.legend(frameon=False)
plt.savefig("../output/zipf_rank_frequency.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Zipf slope (lemmas, ranks 10-1000): {slope:.3f}")

cum = np.cumsum(freqs) / freqs.sum()
top = lemma_counts.head(5)
print("top lemmas:", {f"{l} [{bw_to_arabic(l)}]": int(n) for l, n in top.items()})
for k in [10, 100, 1000]:
    print(f"top {k:>4} lemmas cover {cum[k-1]:.1%} of all 77,915 stem tokens")
print("Saved output/zipf_rank_frequency.png")

Zipf slope (lemmas, ranks 10-1000): -1.103
top lemmas: {'min [مِن]': 3226, '{ll~ah [ٱللَّه]': 2699, 'maA [مَا]': 2565, 'laA [لَا]': 1738, 'fiY [فِى]': 1701}
top   10 lemmas cover 26.1% of all 77,915 stem tokens
top  100 lemmas cover 57.5% of all 77,915 stem tokens
top 1000 lemmas cover 88.9% of all 77,915 stem tokens
Saved output/zipf_rank_frequency.png


## 3. Near-miss sensitivity

For each audited claim: of the methods computed, how many land **exactly** on the claimed
numbers, how many miss by at most 1, by at most 2? (Distance for a pair claim = the larger of
the two sides' misses.) This shows how dense the near-miss space around each celebrated number
is — an exact hit means less when several other selections land within +/-1, and a "does not
hold" verdict is softened or hardened by whether anything came close.

In [6]:
grid = pd.read_csv("../output/claims_audit_grid.csv")

def miss(row):
    d = abs(row.count_a - row.claimed_a)
    if pd.notna(row.claimed_b):
        if pd.isna(row.count_b):
            return np.nan
        d = max(d, abs(row.count_b - row.claimed_b))
    return d

grid["miss"] = grid.apply(miss, axis=1)
g = grid.dropna(subset=["miss"])
near = (g.groupby(["id", "claim"])
          .agg(methods=("miss", "size"),
               exact=("miss", lambda s: int((s == 0).sum())),
               within_1=("miss", lambda s: int((s <= 1).sum())),
               within_2=("miss", lambda s: int((s <= 2).sum())),
               closest_miss=("miss", "min"))
          .reset_index())
near["closest_miss"] = near.closest_miss.astype(int)
display(near.sort_values("id").reset_index(drop=True))
near.to_csv("../output/claims_near_miss.csv", index=False)
print("Saved output/claims_near_miss.csv")

nh = near[near.exact == 0].sort_values("closest_miss")
print("\nClaims with no exact method, by closest miss:")
print(nh[["id", "claim", "closest_miss"]].to_string(index=False))

,id,claim,methods,exact,within_1,within_2,closest_miss
0,C01,Dunya = Akhira,10,5,5,5,0
1,C02,Mala'ika = Shayatin,10,1,1,1,0
2,C03,Hayat = Mawt,11,0,0,0,39
3,C04,Rajul = Imra'a (24),11,1,1,1,0
4,C04b,"Rajul = Imra'a (23, 'chromosome pairs')",11,0,1,1,1
5,C05,Salihat = Sayyi'at,10,0,0,0,13
6,C06,Qul = Qalu,11,1,1,1,0
7,C07,Iblis = isti'adha,8,0,0,2,2
8,C08,Zakat = Baraka,10,0,0,0,17
9,C09,Abrar : Fujjar = 6 : 3,3,1,1,1,0


Saved output/claims_near_miss.csv

Claims with no exact method, by closest miss:
  id                                   claim  closest_miss
C04b Rajul = Imra'a (23, 'chromosome pairs')             1
C17b        Harr = Bard (5, 'summer/winter')             1
 C19                          Nabat = Shajar             1
 C20            Jaza' : Maghfira = 117 : 234             1
 C07                       Iblis = isti'adha             2
 C15                          Musiba = Shukr             2
 C13                       Muhammad = Sharia             3
 C10                    Yusr : Usr = 36 : 12             7
 C05                      Salihat = Sayyi'at            13
 C08                          Zakat = Baraka            17
 C03                            Hayat = Mawt            39
 C12                           Nas = Anbiya'            50


## 4. Sanity checks and deferred work

Pinned invariants (mirrored in `tests/test_base_rates.py`). Deferred analyses are tracked in
the beads issue tracker: `quran-frequencies-dk1` (Monte Carlo null model) and
`quran-frequencies-0bb` (cross-annotation-scheme comparison).

In [7]:
assert len(lemma_counts) == 4832 and len(root_counts) == 1642
assert hapax == 1994
assert int(mult.get(88, 0)) == 4 and int(mult.get(25, 0)) == 12 and int(mult.get(115, 0)) == 1
assert equal_pairs(lemma_counts, 10) == 8594
assert equal_pairs(root_counts, 10) == 2817
jaza = near[near.id == "C20"].iloc[0]
assert jaza.exact == 0 and jaza.closest_miss == 1   # jaza'/maghfira misses 117:234 by exactly 1
print("All base-rate pins verified.")

All base-rate pins verified.
